# LLM の出力に透かしを入れて検出する（transformers の WatermarkingConfig と SynthID）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_llm_analysis/llm_watermark.ipynb)

NLP クックブック 17-10 ／ LLM の評価と分析

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

自分たちが公開する LLM の生成文に、読んでもわからない「透かし」（ウォーターマーク）を入れておき、あとから文章だけを見て、自分のモデルが生成したものかを判定したい。transformers に組み込みの透かしと検出器を使い、検出のしやすさ、文章の質、短い文章や言い換えへの強さを確かめる。

In [1]:
# Colab の T4 で 2〜3 分程度 (モデルのダウンロードを含む)
import math
import re
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import transformers
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer, WatermarkDetector, WatermarkingConfig

print("transformers", transformers.__version__, "/ torch", torch.__version__)   # 検出には、生成時と同じ版で同じ乱数の表を作れることが要る
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"   # SB Intuitions、MIT ライセンス

def load_tokenizer(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    # transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
    # 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
    tokenizer.backend_tokenizer.model = models.Unigram(
        [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
    tokenizer.padding_side = "left"
    return tokenizer

tokenizer = load_tokenizer(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, device_map="auto").eval()
DEVICE = model.device

KEY = 73519   # 透かしの鍵。実際には乱数で選んで秘密にする (hashing_key の既定値 15485863 は公開されているので使わない)
wm = WatermarkingConfig(greenlist_ratio=0.25, bias=2.0, hashing_key=KEY, seeding_scheme="lefthash")   # greenlist_ratio が γ、bias が δ

TOPICS = ["猫", "富士山", "コーヒー", "地震", "梅雨", "将棋", "新幹線", "お盆", "俳句", "温泉", "桜", "寿司", "台風", "図書館", "自転車",
          "歌舞伎", "火山", "味噌", "電子レンジ", "マラソン", "民泊", "花粉症", "城", "相撲", "お茶", "漫画", "雪", "米作り", "神社", "地下鉄",
          "太陽光発電", "折り紙", "盆栽", "海水浴", "コンビニ", "ラーメン", "囲碁", "書道", "花火", "動物園", "灯台", "鉄道模型", "紅葉", "納豆",
          "宇宙ステーション", "博物館", "天気予報", "選挙", "郵便", "高齢化", "リサイクル", "睡眠", "ワクチン", "プログラミング", "翻訳", "辞書",
          "敬語", "方言", "水族館", "キャンプ"]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": f"「{t}」について、400 字程度で説明してください。"}],
                                         add_generation_prompt=True, tokenize=False) for t in TOPICS]

@torch.no_grad()
def generate(watermarking_config=None, seed=0, max_new_tokens=200, **kwargs):
    torch.manual_seed(seed)
    enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
    # モデルの generation_config (temperature 0.7, top_p 0.9, repetition_penalty 1.05) のうち温度だけを使い、ほかは打ち消す (top_p は下で比べる)
    sampling = dict(do_sample=True, temperature=0.7, top_p=1.0, top_k=0, repetition_penalty=1.0) | kwargs
    out = model.generate(**enc, watermarking_config=watermarking_config, max_new_tokens=max_new_tokens,
                         pad_token_id=tokenizer.pad_token_id, **sampling)
    gen = out[:, enc["input_ids"].shape[1]:]
    return tokenizer.batch_decode(gen, skip_special_tokens=True), gen

def encode(texts, tok=tokenizer):
    return [tok(t, add_special_tokens=False)["input_ids"] for t in texts]

def kgw_z(id_lists, config, device=DEVICE, ignore_repeated_ngrams=True):
    """文章のトークン列だけから green のトークンの数を数え、z を返す。モデルの重みもプロンプトも要らないが、
    透かしの設定 (鍵を含む)、語彙数、トークナイザー、デバイスは生成時と同じにする。
    ignore_repeated_ngrams=True で、同じ n-gram (green を決める文脈と候補の組) は何度出ても 1 回だけ数える
    (原著の実装 jwkirchenbauer/lm-watermarking が検出で必ず使うよう勧める設定。transformers の既定は False)"""
    detector = WatermarkDetector(model_config=model.config, device=device, watermarking_config=config,
                                 ignore_repeated_ngrams=ignore_repeated_ngrams)
    return np.array([detector(torch.tensor([ids], device=device), return_dict=True).z_score[0] if len(ids) > config.context_width + 1
                     else np.nan for ids in id_lists])

plain, _ = generate()                     # 透かしなし
marked, marked_ids = generate(wm)         # 透かしあり (γ = 0.25, δ = 2)
for label, text in [("透かしなし", plain[0]), ("透かしあり", marked[0])]:
    print(f"[{label}] z = {kgw_z(encode([text]), wm)[0]:.2f}\n{text[:150]}…\n")

transformers 5.22.1 / torch 2.14.1+cu130
[透かしなし] z = 1.01
「猫」は、一般的に丸みを帯びた耳と長い尾を持つ社交的で playfulな動物として知られています。特にペットとして広く飼育されており、その愛嬌のある外観と gentle な性格が多くの飼い主を魅了します。猫の起源は古代エジプトにまで遡り、神話や芸術にも頻繁に登場します。狩猟能力に長けており、獲物を捕…

[透かしあり] z = 4.54
猫、特にdomestic cat（飼い猫）は、世界中で beloved pet（ belovedペット）として広く愛されています。猫は一般的に社交的で、人間とのコミュニケーションを好む傾向にあります。彼らはIntelligent（intelligent、知恵のある）creaturesであり、忙しく動…



In [2]:
# 60 件ずつの z の分布を比べる。z > 4 なら「透かしあり」と判定する (透かしのない文章が偶然 z > 4 になる確率は約 3e-5)
z_plain, z_marked = kgw_z(encode(plain), wm), kgw_z(encode(marked), wm)
print(pd.DataFrame({"透かしなし": z_plain, "透かしあり": z_marked}).describe().round(2).T.to_string())
print(f"z > 4 の割合: 透かしなし {np.mean(z_plain > 4):.2f} / 透かしあり {np.mean(z_marked > 4):.2f}")
z_rep = kgw_z(encode(plain), wm, ignore_repeated_ngrams=False)
print(f"繰り返す n-gram も毎回数える (transformers の既定) と: 透かしなしの z の平均 {z_rep.mean():.2f}, 最大 {z_rep.max():.2f}")

# 検出は文章をトークン化し直して行う。生成したトークン列のまま数えた場合や、直さない (transformers 5 のまま) トークナイザーと比べる
gen_ids = [[i for i in row.tolist() if i not in (tokenizer.pad_token_id, tokenizer.eos_token_id)] for row in marked_ids]
raw_ids = encode(marked, AutoTokenizer.from_pretrained(name))
z_raw = kgw_z(raw_ids, wm)
print(f"z の平均: 生成したトークン列 {np.nanmean(kgw_z(gen_ids, wm)):.2f} / 直したトークナイザー {z_marked.mean():.2f} / transformers 5 のまま {z_raw.mean():.2f}")
print(f"z > 4 の割合: 直したトークナイザー {np.mean(z_marked > 4):.2f} / transformers 5 のまま {np.mean(z_raw > 4):.2f}")
print(f"生成と同じトークン列に戻った文章の割合: 直したトークナイザー {np.mean([g == f for g, f in zip(gen_ids, encode(marked))]):.2f}"
      f" / transformers 5 のまま {np.mean([g == r for g, r in zip(gen_ids, raw_ids)]):.2f}")

# 設定やデバイスが生成時と違うと、green の表が変わって検出できない
print(f"鍵が違う: z の平均 {kgw_z(encode(marked[:20]), WatermarkingConfig(greenlist_ratio=0.25, bias=2.0, hashing_key=KEY + 1)).mean():.2f}")
print(f"CPU で検出 (生成は {DEVICE}): z の平均 {kgw_z(encode(marked[:20]), wm, device='cpu').mean():.2f}")

       count  mean   std   min   25%   50%   75%   max
透かしなし   60.0 -0.08  1.00 -2.39 -0.74 -0.13  0.59  2.87
透かしあり   60.0  5.16  1.04  2.33  4.54  5.25  5.97  7.27
z > 4 の割合: 透かしなし 0.00 / 透かしあり 0.83
繰り返す n-gram も毎回数える (transformers の既定) と: 透かしなしの z の平均 -0.28, 最大 2.29
z の平均: 生成したトークン列 5.24 / 直したトークナイザー 5.16 / transformers 5 のまま 3.91
z > 4 の割合: 直したトークナイザー 0.83 / transformers 5 のまま 0.50
生成と同じトークン列に戻った文章の割合: 直したトークナイザー 0.58 / transformers 5 のまま 0.00
鍵が違う: z の平均 -0.58
CPU で検出 (生成は cuda:0): z の平均 0.24


In [3]:
# 透かしの強さ (δ、γ、方式) と、検出のしやすさ・文章の質 (大きなモデルで測ったパープレキシティ) の釣り合いを見る。
# 幅 (context_width) の意味は方式で違う (transformers 5.22.1 の実装): selfhash では候補のトークン自身を含む末尾の「幅」個のトークンから
# green が決まる (幅 4 なら直前の 3 トークン + 候補)。lefthash では幅を変えても直前の 1 トークンしか使わない。
# SynthID Text (Google DeepMind) も同じ条件で比べる。Colab の T4 で 10〜15 分程度 (3B のモデルのダウンロードを含む)
from transformers import SynthIDTextWatermarkingConfig

# 鍵は 20 個 (深さ 20)。実際には大きな乱数を選んで秘密にする
synthid = SynthIDTextWatermarkingConfig(keys=np.random.default_rng(KEY).integers(0, 2**31, size=20).tolist(), ngram_len=5)

def synthid_z(id_lists, config=synthid, device=DEVICE):
    """SynthID の g 値 (直前の 4 トークン、候補のトークン、鍵から決まる 0/1) の平均を z にする。透かしがなければ g はほぼ確率 0.5 で 1。
    transformers の SynthIDTextWatermarkDetector は学習したベイズ検出器を使うので、ここでは学習の要らない平均値の検定にした"""
    proc = config.construct_processor(model.config.vocab_size, device)
    p = proc.sampling_table.float().mean().item()            # g が 1 になる確率 (乱数の表から決まる)
    zs = []
    for ids in id_lists:
        x = torch.tensor([ids], device=device)
        if x.shape[1] <= config.ngram_len:
            zs.append(np.nan)
            continue
        g = proc.compute_g_values(x)[0].float()               # (n-gram の数, 鍵の数)
        g = g[proc.compute_context_repetition_mask(x)[0]]      # 同じ文脈の繰り返しは、生成時に透かしを入れていないので数えない
        zs.append((g.mean().item() - p) / math.sqrt(p * (1 - p) / g.numel()))
    return np.array(zs)

def kgw(gamma=0.25, delta=2.0, scheme="lefthash", width=1):
    return WatermarkingConfig(greenlist_ratio=gamma, bias=delta, hashing_key=KEY, seeding_scheme=scheme, context_width=width)

def detector_for(config):
    """生成の設定に合った検出の関数を返す。透かしなし (None) の文章は、γ=0.25 δ=2 の検出器で測る"""
    if isinstance(config, SynthIDTextWatermarkingConfig):
        return synthid_z
    return lambda ids: kgw_z(ids, config or wm)

settings = {   # 名前: (生成の設定, generate に渡す追加の引数)
    "なし": (None, {}),
    "KGW γ=0.25 δ=1": (kgw(delta=1.0), {}),
    "KGW γ=0.25 δ=2": (wm, {}),
    "KGW γ=0.25 δ=4": (kgw(delta=4.0), {}),
    "KGW γ=0.25 δ=8": (kgw(delta=8.0), {}),
    "KGW γ=0.5 δ=2": (kgw(gamma=0.5), {}),
    "KGW γ=0.25 δ=2 selfhash 幅4": (kgw(scheme="selfhash", width=4), {}),
    "KGW γ=0.25 δ=2 top_p=0.9": (wm, {"top_p": 0.9}),
    "SynthID": (synthid, {}),
}
texts_by = {"なし": plain, "KGW γ=0.25 δ=2": marked}
for label, (config, kwargs) in settings.items():       # 先に 0.5B のモデルで全部生成してから、3B のモデルを読み込む (GPU メモリを節約する)
    if label not in texts_by:
        texts_by[label] = generate(config, **kwargs)[0]
print(f"生成までの GPU メモリの最大: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

oracle_name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 評価と言い換え用の大きなモデル (SB Intuitions、MIT ライセンス)
oracle_tok = load_tokenizer(oracle_name)
oracle = AutoModelForCausalLM.from_pretrained(oracle_name, dtype=DTYPE, device_map="auto").eval()

@torch.no_grad()
def perplexity(prompt, text):
    """プロンプトを条件にしたときの、生成部分のパープレキシティ"""
    p = oracle_tok(prompt, add_special_tokens=False, return_tensors="pt")["input_ids"]
    c = oracle_tok(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    ids = torch.cat([p, c], dim=1).to(oracle.device)
    logp = oracle(ids).logits[0, p.shape[1] - 1:-1].float().log_softmax(-1)
    return math.exp(-logp.gather(1, ids[0, p.shape[1]:, None]).mean().item())

rows = []
for label, (config, kwargs) in settings.items():
    texts = texts_by[label]
    detect = detector_for(config)
    z, z_null = detect(encode(texts)), detect(encode(plain))      # 同じ検出器で、透かしのない文章を誤検出しないかも見る
    ppl = np.array([perplexity(p, t) for p, t in zip(prompts, texts)])
    rows.append({"透かし": label, "z の平均": z.mean(), "z の最小": z.min(), "z>4 の割合": np.mean(z > 4),
                 "透かしなしで z>4": np.mean(z_null > 4), "PPL の中央値": np.median(ppl), "例": texts[1][:25]})
print(pd.DataFrame(rows).round(2).to_string(index=False))
# selfhash を既定の幅 1 で使うと、green かどうかが各トークンだけで決まってしまう。透かしのない文章で確かめる
for ignore in [False, True]:
    z_null = kgw_z(encode(plain), kgw(scheme="selfhash"), ignore_repeated_ngrams=ignore)
    print(f"selfhash 幅1 の検出器 (ignore_repeated_ngrams={ignore}) に透かしなしの文章を入れると: z の平均 {z_null.mean():.2f}, z>4 の割合 {np.mean(z_null > 4):.2f}")

生成までの GPU メモリの最大: 7.3 GB
                       透かし  z の平均  z の最小  z>4 の割合  透かしなしで z>4  PPL の中央値                         例
                        なし  -0.08  -2.39     0.00         0.0      9.48 富士山は、日本アルプスを構成する重要な山の一つであ
            KGW γ=0.25 δ=1   2.15  -0.38     0.05         0.0     10.09 富士山は日本を代表する山であり、そのmajesti
            KGW γ=0.25 δ=2   5.16   2.33     0.83         0.0     12.02 富士山は日本を代表する山であり、そのmajesti
            KGW γ=0.25 δ=4  10.54   7.16     1.00         0.0     20.30 富士山は日本を代表する山であり、その標高3,776
            KGW γ=0.25 δ=8  17.02  12.77     1.00         0.0     78.46 富士山は日本を代表する山であり、その標高3,776
             KGW γ=0.5 δ=2   4.47   1.91     0.70         0.0     11.71 富士山は日本を代表する山であり、そのmajesti
KGW γ=0.25 δ=2 selfhash 幅4   6.29   2.52     0.93         0.0     10.49 富士山は日本を代表する最高峰の山であり、その名は広
  KGW γ=0.25 δ=2 top_p=0.9   4.04   1.24     0.57         0.0      8.33 富士山は日本を代表する山であり、そのmajesti
                   SynthID   7.63   5.26     1.00         0.0      9.49 富士山は、日本を代表する山の一つであり、そ

In [4]:
# 頑健さ (1): 文章を短くする。green の割合が同じでも、数えるトークン数 T が少ないと z (∝ √T) が小さく、検出できない
detectors = {k: detector_for(settings[k][0]) for k in ["KGW γ=0.25 δ=2", "SynthID"]}
for n_chars in [25, 50, 100, 200, None]:
    n_tok = np.mean([len(ids) for ids in encode([t[:n_chars] for t in marked])])
    res = [f"{k} z>4 の割合 {np.mean(f(encode([t[:n_chars] for t in texts_by[k]])) > 4):.2f}" for k, f in detectors.items()]
    print(f"先頭 {n_chars or '全部'} 文字 (KGW の文章で平均 {n_tok:.0f} トークン): " + " / ".join(res))

先頭 25 文字 (KGW の文章で平均 11 トークン): KGW γ=0.25 δ=2 z>4 の割合 0.02 / SynthID z>4 の割合 0.02
先頭 50 文字 (KGW の文章で平均 22 トークン): KGW γ=0.25 δ=2 z>4 の割合 0.02 / SynthID z>4 の割合 0.03
先頭 100 文字 (KGW の文章で平均 44 トークン): KGW γ=0.25 δ=2 z>4 の割合 0.05 / SynthID z>4 の割合 0.33
先頭 200 文字 (KGW の文章で平均 88 トークン): KGW γ=0.25 δ=2 z>4 の割合 0.17 / SynthID z>4 の割合 0.83
先頭 全部 文字 (KGW の文章で平均 192 トークン): KGW γ=0.25 δ=2 z>4 の割合 0.83 / SynthID z>4 の割合 1.00


In [5]:
# 頑健さ (2): 透かしのない別のモデル (3B) で、文を 1 つずつ言い換える。半分の文だけ言い換えた場合と、全部言い換えた場合を比べる。T4 で 5〜8 分程度
@torch.no_grad()
def paraphrase(sentences, batch_size=32):
    outs = []
    for i in range(0, len(sentences), batch_size):
        batch = [oracle_tok.apply_chat_template([{"role": "user", "content": f"次の文を、意味を変えずに別の言葉で言い換えてください。言い換えた文だけを出力してください。\n\n{s}"}],
                                                add_generation_prompt=True, tokenize=False) for s in sentences[i:i + batch_size]]
        enc = oracle_tok(batch, return_tensors="pt", padding=True, add_special_tokens=False).to(oracle.device)
        out = oracle.generate(**enc, max_new_tokens=120, do_sample=False, repetition_penalty=1.0, pad_token_id=oracle_tok.pad_token_id)
        outs += [s.strip().split("\n")[0] for s in oracle_tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return outs

def attack(texts):
    split = [re.findall(r"[^。]+。?", t) for t in texts]          # 句点で文に分ける
    para = iter(paraphrase([s for ss in split for s in ss]))
    para_split = [[next(para) for _ in ss] for ss in split]
    half = ["".join(p if i % 2 else s for i, (s, p) in enumerate(zip(ss, ps))) for ss, ps in zip(split, para_split)]   # 2 文目、4 文目… だけ
    return half, ["".join(ps) for ps in para_split]

detectors["KGW γ=0.25 δ=2 selfhash 幅4"] = detector_for(settings["KGW γ=0.25 δ=2 selfhash 幅4"][0])
rows = []
for k, f in detectors.items():
    half, full = attack(texts_by[k])
    if k == "KGW γ=0.25 δ=2":
        print("元の文章:", texts_by[k][2][:100], "\n全部言い換え:", full[2][:100], "\n")
    for label, texts in [("言い換えなし", texts_by[k]), ("半分の文を言い換え", half), ("全部の文を言い換え", full), ("(参考) 透かしなし", plain)]:
        z = f(encode(texts))
        rows.append({"透かし": k, "文章": label, "z の平均": z.mean(), "z>4 の割合": np.mean(z > 4)})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print(f"GPU メモリの最大使用量: テンソル {torch.cuda.max_memory_allocated() / 1e9:.1f} GB / PyTorch が確保した領域 {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

元の文章: コーヒーは、コーヒーノキと呼ばれる植物の種子から生成される飲料であり、世界中で広く愛飲されています。その起源はエチオピアにさかのぼり、アラビア半島を経てヨーロッパやアメリカへと広まっていきました。コー 
全部言い換え: コーヒーは、コーヒーノキの種子を原料とする飲料で、世界中で広く親しまれています。その起源はエチオピアにあり、アラビア半島を経由してヨーロッパやアメリカへと伝播しました。コーヒーには多様な種類があり、特 

                       透かし         文章  z の平均  z>4 の割合
            KGW γ=0.25 δ=2     言い換えなし   5.16     0.83
            KGW γ=0.25 δ=2  半分の文を言い換え   3.60     0.32
            KGW γ=0.25 δ=2  全部の文を言い換え   1.66     0.02
            KGW γ=0.25 δ=2 (参考) 透かしなし  -0.08     0.00
                   SynthID     言い換えなし   7.63     1.00
                   SynthID  半分の文を言い換え   4.04     0.58
                   SynthID  全部の文を言い換え   0.50     0.00
                   SynthID (参考) 透かしなし   0.25     0.00
KGW γ=0.25 δ=2 selfhash 幅4     言い換えなし   6.29     0.93
KGW γ=0.25 δ=2 selfhash 幅4  半分の文を言い換え   4.67     0.77
KGW γ=0.25 δ=2 selfhash 幅4  全部の文を言い換え   2.78     0.15
KGW γ=0.25 δ=2 selfhash 幅4 (参考) 透かしなし   0.90     0.00
GPU メモリの最大使用量: テンソル 9.4 GB / PyTorch が確保した領域 10.1 GB


- Kirchenbauer らの方式 (Kirchenbauer et al., 2023, "A Watermark for Large Language Models")。直前のトークンと鍵から語彙の γ を「green」に選び、そのロジットに δ を足して生成する。検出には鍵と設定とトークナイザーがあればよく、モデルの重みもプロンプトも要らない。`generate(watermarking_config=...)` で透かしを入れ、`WatermarkDetector` が green の数から z を計算する（`prediction` の既定のしきい値は z > 3 で、透かしのない文章が偶然超える確率は約 1e-3。ここでは原論文にならって z > 4 にした）。検出では `ignore_repeated_ngrams=True` を付け、同じ n-gram を何度も数えないようにした（原著の実装 jwkirchenbauer/lm-watermarking の README が「検出では必ず使う」と勧める設定で、transformers の既定は False）。60 件（約 200 トークン）で、透かしなしの z は平均 −0.08（最大 2.87）、γ = 0.25、δ = 2 の透かしありは平均 5.16 で、z > 4 は 83% だった（繰り返しも数える既定のままだと、透かしなしは平均 −0.28、最大 2.29）。60 件なので、割合には ±10 ポイント程度の幅がある。
- 検出の条件は生成時とそろえる。鍵を 1 つ変えると z の平均は −0.58、GPU で生成した文章を CPU で検出すると 0.24 になり、まったく検出できなかった（green の表を `torch.Generator` の乱数で作るので、デバイスで変わる）。CPU と GPU の違いしか確かめておらず、GPU の種類が違う場合（たとえば T4 と検証に使った GPU）に同じ表になるかは確かめていない。上の数値も、GPU によって変わりうる。鍵は秘密にし、transformers と torch の版も記録して固定しておく（版を変えても同じ表になるかも確かめていない）。トークナイザーも同じにする: sarashina2.2 のトークナイザーを直さずに使うと z の平均は 3.91、z > 4 は 50% まで下がった（直し方は、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照）。直したトークナイザーでは 5.16 で、生成したトークン列のままの 5.24 とほとんど変わらないが、同じトークン列に戻った文章は 58% で、同じ文字列に分割が複数ありうる分だけ目減りは残る。
- δ を大きくすると検出しやすくなるが、文章の質が落ちる。γ = 0.25 で δ を 0, 1, 2, 4, 8 と上げると、z の平均は −0.08, 2.15, 5.16, 10.54, 17.02、3B のモデルで測ったパープレキシティの中央値は 9.5, 10.1, 12.0, 20.3, 78.5 になった（同じ系列のモデルで測ったので目安）。δ = 1 では z > 4 は 5% で、200 トークンではほとんど検出できない。γ = 0.5 は γ = 0.25 より z が低かった (4.47)。transformers では（5.22.1 で確認）透かしの処理が top_p などの絞り込みのあとにかかるので、モデルの既定の top_p = 0.9 のまま生成すると、δ = 2 でも z の平均は 4.04（z > 4 は 57%）に下がった。この設定のパープレキシティ (8.3) が透かしなし (9.5) より低いのは top_p で候補を絞った効果で、透かしのコストとは比べられない。
- `seeding_scheme="selfhash"` は名前が同じでも、原著の実装と transformers で中身が違う。原著の README は幅 (context width) h = 4 を勧め、`selfhash` を幅 4 の方式の略称としている。transformers の `selfhash` は既定が `context_width=1` で、そのままだと green かどうかが各トークンだけで決まる。そのため、繰り返しも数える既定の検出器では、透かしのない文章でも z の平均が 3.65、z > 4 が 40% になった（`ignore_repeated_ngrams=True` にすると 0.49 と 0%）。selfhash を使うなら `context_width=4` を明示する（候補のトークン自身を含む 4 トークンから green が決まる。lefthash では幅を変えても直前の 1 トークンしか使わない）。幅 4 では、透かしのない文章の z > 4 は 0（ただし z の平均は 0.90 とやや高い）、透かしありは z の平均 6.29（z > 4 は 93%）、パープレキシティの中央値 10.5 だった。候補を 1 つずつ確かめながら生成するので、lefthash より生成が遅い。
- SynthID Text (Dathathri et al., 2024, Nature, Google DeepMind) は、直前の数トークン、候補のトークン、複数の鍵から決まる 0/1 の値（g 値）が大きいトークンを、トーナメント方式の抽出で選ぶ。ここでは z の平均 7.63（z > 4 は 100%）と検出しやすく、パープレキシティの中央値も 9.5 で、透かしなし (9.5) とほとんど変わらなかった。transformers の公式の検出器 `SynthIDTextWatermarkDetector` は、学習したベイズ検出器 (`BayesianDetectorModel`) を使う。学習の手順は transformers-research-projects リポジトリにあり、ここでは代わりに、g 値の平均の検定を小さな関数で書いた。transformers の実装はハッシュの方式が Gemini アプリのものと違うので（ソースコードのコメント）、Gemini の出力の検出には使えない。
- 短い文章では検出できない。z は green の割合が同じでも √T に比例するので、先頭 100 文字（約 44 トークン）に切ると z > 4 は KGW (δ = 2) で 5%、SynthID で 33%、200 文字（約 88 トークン）で 17% と 83% だった。また、次のトークンがほぼ決まっている箇所（エントロピーが低い箇所）には透かしが入りにくい（Kirchenbauer らの原論文）。
- 言い換えに弱い。透かしのない 3B のモデルで半分の文を言い換えると、z > 4 は KGW（lefthash、幅 1）で 32%、SynthID で 58%、selfhash 幅 4 で 77%、全部の文を言い換えると 2%、0%、15% になった。同じグループの続編の研究 (Kirchenbauer et al., ICLR 2024, "On the Reliability of Watermarks for Large Language Models") は、言い換えた文章にも元の断片が残るので、長くすれば検出できると報告している。透かしは生成する側が協力して入れておく仕組みで、透かしのない文章を文章の統計だけで見分ける方法は、この章の「LLM が生成した文章を見分ける（パープレキシティ、Binoculars、分類器）」を参照。
- transformers は Apache-2.0 ライセンスで、Colab に入っている版をそのまま使った（検証は 5.22.1）。透かしの仕組み（green の選び方と z 検定）を自分で実装して確かめる版は、付録「仕組みを理解するための実装」の「LLM の出力に入れる透かし（green リスト）を自分で実装し、統計的に検出する」にある。